# 🔥 Fire Detection Demo (Drive Model)

This notebook loads your trained model from Google Drive and launches the inference interface.

**Pre-requisites:**
1. Store your `yolov5s_best.pt` in your Google Drive (e.g., inside a `fire_detection/model/` folder).

In [ ]:
# 1. Install Dependencies
%pip install ultralytics gradio

In [1]:
# 2. Mount Drive & Locate Model
import os
from ultralytics import YOLO

# Check Colab Environment
is_colab = os.path.exists('/content')

if is_colab:
    print("Mounting Google Drive... (Please authorize)")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # --- CONFIGURE YOUR DRIVE PATH HERE ---
    # Common path based on previous context, user can edit this:
    drive_model_path = '/content/drive/MyDrive/fire_detection/yolov5-fire-detection/model/yolov5s_best.pt'
    
    # Check if the specific path exists
    if os.path.exists(drive_model_path):
        model_path = drive_model_path
        print(f"✅ Model found in Drive: {model_path}")
    else:
        # Search fallback: Look for file in root MyDrive if folder structure differs
        print(f"⚠️ Model not found at expected path: {drive_model_path}")
        print("Searching MyDrive for 'yolov5s_best.pt'...")
        fallback_path = '/content/drive/MyDrive/yolov5s_best.pt'
        
        if os.path.exists(fallback_path):
            model_path = fallback_path
            print(f"✅ Verified model found at root: {model_path}")
        else:
            print("❌ Model not found in verified paths. Using auto-downloaded base model for demo.")
            model_path = 'yolov5su.pt'
else:
    # Local fallback path
    model_path = '../model/yolov5s_best.pt'

print(f"Loading model from: {model_path}")
model = YOLO(model_path)

Mounting Google Drive... (Please authorize)


ValueError: mount failed

In [ ]:
# 3. Run Gradio Interface
import gradio as gr
import cv2
from PIL import Image

def detect_image(image):
    if image is None: return None
    results = model(image)
    return results[0].plot()

def detect_video(video_path):
    if video_path is None: return None
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    output_path = "output_detected.mp4"
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        results = model(frame)
        out.write(results[0].plot())
    cap.release()
    out.release()
    return output_path

with gr.Blocks() as demo:
    gr.Markdown(f"# 🔥 Fire Detection (Model: `{os.path.basename(model_path)}`)")
    with gr.Tab("Image"):
        gr.Interface(detect_image, gr.Image(type="pil"), gr.Image())
    with gr.Tab("Video"):
        gr.Interface(detect_video, gr.Video(), gr.Video())

demo.launch(share=True)